# 10 — Hyperparameter tuning (P3)

**Goal (roadmap P3):** `make_baseline_model`'s `HistGradientBoostingRegressor` is
still at the official starter notebook's original hyperparameters
(`learning_rate=0.05, max_iter=300, max_depth=8, min_samples_leaf=50,
l2_regularization=1.0`), chosen for a much smaller feature set than this project's
current one (neighbourhood, climatology, anchor-age, SPEI_12 prior-reading -
14 columns vs. the starter's ~4-8). Per `structuring-ml-projects` rule 8, tuning
comes after feature work, using random search over the existing evaluation harness.

**Sources** (checked before writing any search code, per this project's
methodology correction on 2026-09-05 - see README):
- Bergstra & Bengio, *Random Search for Hyper-Parameter Optimization* (JMLR 2012) -
  random search over grid search for this multi-dimensional space.
- `hands-on-ml` skill, ch07 "Ensemble Learning and Random Forests" - flags manually
  grid-searching a boosting model's tree count as wasteful; the fix is
  `early_stopping`/`staged_predict`-style tools instead. `HistGradientBoostingRegressor`
  already auto-enables early stopping for datasets this large (>10,000 rows) - so
  `max_iter` is fixed to a generous ceiling (1000) and NOT searched directly; early
  stopping picks the actual tree count per candidate. Also confirms `learning_rate`
  and tree count are "a paired tuning surface", and lists the standard boosting
  levers this search covers.
- `real-world-ml` skill, ch04 "Model Evaluation and Optimization" - confirms the
  standard boosting tuning-parameter set (learning rate, max depth, min samples per
  leaf, tree count) and the grid-search refinement rule used below to read results:
  "boundary optimum -> expand; high sensitivity -> densify/log-scale; low
  sensitivity -> coarsen."


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from src import config, data, evaluate, features, model
from src.train import get_feature_cols

pd.set_option("display.width", 120)
raw_train, test, _ = data.load_raw_data()
target_horizons = evaluate.compute_test_horizons(raw_train, test)
masked_month_fraction, masked_row_fraction = evaluate.measure_masking_pattern(test)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## Search space

Standard boosting levers per the sources above, sampled log-uniformly where the
parameter's effect is naturally multiplicative (learning rate, min-samples-leaf,
l2 regularization) and uniformly over integers otherwise (max depth, max leaf
nodes). `max_iter` is fixed at a generous ceiling (1000) - `early_stopping='auto'`
(the default, active here since fit sets are always >10,000 rows) picks the actual
count per candidate, avoiding the "wasteful" brute-force retraining `hands-on-ml`
ch07 warns against.


In [2]:
def sample_configs(n, seed):
    rng = np.random.RandomState(seed)
    configs = []
    for _ in range(n):
        configs.append(dict(
            learning_rate=float(10 ** rng.uniform(np.log10(0.01), np.log10(0.3))),
            max_depth=int(rng.randint(3, 13)),
            min_samples_leaf=int(10 ** rng.uniform(np.log10(10), np.log10(200))),
            l2_regularization=float(10 ** rng.uniform(np.log10(0.01), np.log10(10))),
            max_leaf_nodes=int(rng.randint(15, 128)),
        ))
    return configs


def make_model(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("gbr", HistGradientBoostingRegressor(
            loss="squared_error",
            max_iter=1000,
            random_state=config.RANDOM_STATE,
            **params,
        )),
    ])


N_CANDIDATES = 25
candidates = sample_configs(N_CANDIDATES, seed=config.RANDOM_STATE)
print(f"{N_CANDIDATES} candidates sampled.")


25 candidates sampled.


## Screening pass (single seed)

25 candidates x 5-seed evaluation would be far too expensive - screen each
candidate on a single masking realisation of `mask_augmented_horizon_matched_split`
first (same harness `src/train.py` uses, just one seed instead of five), then
confirm only the best candidates with the full 5-seed mean before comparing
against the current default. This mirrors `real-world-ml` ch04's own
holdout-vs-k-fold trade-off: a single split is fast but noisy, so it's used only
to narrow the field, never as the final answer.


In [3]:
fit_df, val_df = evaluate.mask_augmented_horizon_matched_split(
    raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
    fit_seed=0, val_seed=100,
)
feature_cols = get_feature_cols(fit_df)
X_fit = features.select_base_features(fit_df, feature_cols)
y_fit = fit_df[config.TARGET_COL].to_numpy()
X_val = features.select_base_features(val_df, feature_cols)
y_val = val_df[config.TARGET_COL].to_numpy()

screening_results = []
for i, params in enumerate(candidates):
    m = make_model(params)
    m.fit(X_fit, y_fit)
    rmse = evaluate.rmse(y_val, model.predict(m, X_val))
    screening_results.append({**params, "rmse": rmse})
    print(f"[{i+1}/{N_CANDIDATES}] rmse={rmse:.4f} params={params}")

screening_df = pd.DataFrame(screening_results).sort_values("rmse").reset_index(drop=True)
screening_df


C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

[1/25] rmse=0.7354 params={'learning_rate': 0.03574712922600243, 'max_depth': 10, 'min_samples_leaf': 60, 'l2_regularization': 0.029380279387035357, 'max_leaf_nodes': 97}


[2/25] rmse=0.7244 params={'learning_rate': 0.014049959523783894, 'max_depth': 10, 'min_samples_leaf': 27, 'l2_regularization': 0.026828750938254375, 'max_leaf_nodes': 17}


[3/25] rmse=0.7257 params={'learning_rate': 0.010725209743171996, 'max_depth': 4, 'min_samples_leaf': 86, 'l2_regularization': 6.541210527692732, 'max_leaf_nodes': 16}


[4/25] rmse=0.7337 params={'learning_rate': 0.018559980846490572, 'max_depth': 7, 'min_samples_leaf': 63, 'l2_regularization': 0.6838478430964042, 'max_leaf_nodes': 122}


...

[22/25] rmse=0.7376 params={'learning_rate': 0.053628539644418466, 'max_depth': 11, 'min_samples_leaf': 84, 'l2_regularization': 0.021511587551819596, 'max_leaf_nodes': 91}


[23/25] rmse=0.7299 params={'learning_rate': 0.01985929016050209, 'max_depth': 9, 'min_samples_leaf': 41, 'l2_regularization': 0.4895834359555105, 'max_leaf_nodes': 18}


[24/25] rmse=0.7758 params={'learning_rate': 0.21907142272152816, 'max_depth': 9, 'min_samples_leaf': 61, 'l2_regularization': 0.4164120360093465, 'max_leaf_nodes': 43}


[25/25] rmse=0.7269 params={'learning_rate': 0.021775224101934065, 'max_depth': 9, 'min_samples_leaf': 23, 'l2_regularization': 0.030455368715396777, 'max_leaf_nodes': 80}


,learning_rate,max_depth,min_samples_leaf,l2_regularization,max_leaf_nodes,rmse
0,0.047171,5,31,8.906204,17,0.722356
1,0.014050,10,27,0.026829,17,0.724433
2,0.046120,3,12,0.085693,102,0.725617
3,0.010725,4,86,6.541211,16,0.725661
4,0.015019,5,97,0.482872,28,0.726505
5,0.037508,4,50,0.575776,103,0.726622
6,0.021775,9,23,0.030455,80,0.726872
7,0.010816,12,11,8.341930,29,0.727973
8,0.160153,3,84,2.347073,19,0.728847
9,0.019859,9,41,0.489583,18,0.729916


## Baseline (current defaults) on the same single-seed split

For a fair screening comparison, one screening cell for the CURRENT
`make_baseline_model` defaults, on this exact same fit/val split.


In [4]:
current_defaults = dict(learning_rate=0.05, max_depth=8, min_samples_leaf=50, l2_regularization=1.0)
baseline_screen_model = model.make_baseline_model()
baseline_screen_model.fit(X_fit, y_fit)
baseline_screen_rmse = evaluate.rmse(y_val, model.predict(baseline_screen_model, X_val))
print(f"Current defaults (max_iter=300 fixed): rmse={baseline_screen_rmse:.4f}")
print(f"Best screened candidate: rmse={screening_df.iloc[0]['rmse']:.4f}")


Current defaults (max_iter=300 fixed): rmse=0.7276
Best screened candidate: rmse=0.7224


## Confirmation pass (5-seed mean) for the top candidates

Per `structuring-ml-projects` step 4: "recompute the baseline in the same run...
this is the final validation before a graduation decision." Re-evaluate the top 3
screened candidates AND the current defaults over the full 5-seed
`mask_augmented_horizon_matched_split` mean (the project's primary honest proxy,
exactly as `src/train.py` reports it) before deciding anything.


In [5]:
def five_seed_mean_rmse(params_or_none):
    rmses = []
    for seed in range(5):
        fit_s, val_s = evaluate.mask_augmented_horizon_matched_split(
            raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
            fit_seed=seed, val_seed=seed + 100,
        )
        cols = get_feature_cols(fit_s)
        Xf = features.select_base_features(fit_s, cols)
        yf = fit_s[config.TARGET_COL].to_numpy()
        Xv = features.select_base_features(val_s, cols)
        yv = val_s[config.TARGET_COL].to_numpy()
        m = model.make_baseline_model() if params_or_none is None else make_model(params_or_none)
        m.fit(Xf, yf)
        rmses.append(evaluate.rmse(yv, model.predict(m, Xv)))
    return rmses


top3 = screening_df.head(3).to_dict("records")
confirmation = {"current_defaults": five_seed_mean_rmse(None)}
for i, row in enumerate(top3):
    params = {k: row[k] for k in ["learning_rate", "max_depth", "min_samples_leaf",
                                    "l2_regularization", "max_leaf_nodes"]}
    confirmation[f"candidate_{i+1}"] = five_seed_mean_rmse(params)

for name, rmses in confirmation.items():
    print(f"{name}: mean={np.mean(rmses):.4f} range=({min(rmses):.4f}, {max(rmses):.4f})  {rmses}")


current_defaults: mean=0.7117 range=(0.6925, 0.7276)  [0.7276082633085404, 0.7222508079865849, 0.7160058108161343, 0.6925121448250914, 0.700238667633396]
candidate_1: mean=0.7155 range=(0.6987, 0.7372)  [0.7223561127933007, 0.73717664233467, 0.7178396119449592, 0.7013519196990388, 0.6987346899811282]
candidate_2: mean=0.7106 range=(0.6957, 0.7244)  [0.7244329494127129, 0.7211689376860094, 0.7140209507741271, 0.6957026818944816, 0.6976487347137154]
candidate_3: mean=0.7141 range=(0.6975, 0.7296)  [0.7256168779370574, 0.7295737381321414, 0.717248641979331, 0.6975398700867467, 0.7005201378601214]


## Round 1 gate decision

Only `candidate_2` (`learning_rate=0.014, max_depth=10, min_samples_leaf=27,
l2_regularization=0.027, max_leaf_nodes=17`) beat the current defaults on the 5-seed
mean (0.7106 vs. 0.7117, ~0.15%, 4/5 seeds) - `candidate_1` (the best-looking
SINGLE-seed screening result, 0.7224) actually got WORSE under 5-seed confirmation
(0.7155), a direct illustration of `real-world-ml` ch04's warning against trusting a
single holdout split. `candidate_2`'s `max_leaf_nodes=17` landed right at the
searched floor (15) - per that same chapter's refinement rule ("boundary optimum ->
expand the grid"), do not accept this as final; see Round 2 below.


## Round 2 — expanded search after a boundary-optimum finding

Round 1's best confirmed candidate (`learning_rate=0.014`) landed with
`max_leaf_nodes=17`, right at the searched floor (15) - per `real-world-ml` ch04's
refinement rule ("boundary optimum -> expand the grid"), re-ran with `learning_rate`
lowered further (loguniform 0.003-0.15, floor cut from 0.01, ceiling cut from 0.3
since high rates were uniformly bad in round 1) and `max_leaf_nodes` widened both
ways (loguniform 5-150). Same methodology as round 1 (25-candidate single-seed
screen, then 5-seed confirmation of the top 3), run as a standalone script for
efficiency (avoids re-running round 1's already-saved 45 fits) - real captured
output, not re-executed live in this notebook, attached as this cell's output.


In [ ]:
# Re-run standalone (see notebooks/10_hyperparameter_tuning.ipynb history / project
# commits for the exact script) - output captured verbatim below.


25 round-2 candidates sampled (refined bounds).
[1/25] rmse=0.7246 params={'learning_rate': 0.004705388772459883, 'max_depth': 4, 'min_samples_leaf': 14, 'l2_regularization': 0.052694933393145266, 'max_leaf_nodes': 15}
[2/25] rmse=0.7284 params={'learning_rate': 0.08645101198577956, 'max_depth': 10, 'min_samples_leaf': 62, 'l2_regularization': 0.6088155292599353, 'max_leaf_nodes': 11}
[3/25] rmse=0.7221 params={'learning_rate': 0.005339426578725584, 'max_depth': 5, 'min_samples_leaf': 40, 'l2_regularization': 7.315173140022737, 'max_leaf_nodes': 33}
[4/25] rmse=0.7269 params={'learning_rate': 0.03810473065946536, 'max_depth': 11, 'min_samples_leaf': 16, 'l2_regularization': 0.17685051085424597, 'max_leaf_nodes': 46}
[5/25] rmse=0.7270 params={'learning_rate': 0.033998037524009435, 'max_depth': 3, 'min_samples_leaf': 31, 'l2_regularization': 7.302379098454287, 'max_leaf_nodes': 22}
[6/25] rmse=0.7261 params={'learning_rate': 0.04120627429631244, 'max_depth': 11, 'min_samples_leaf': 146,

## Round 2 result and real-submission confirmation

All three round-2 candidates beat the current defaults (0.7117) by a clear, mostly
consistent margin (4/5 seeds each) - candidate 1 (`learning_rate=0.0039, max_depth=10,
min_samples_leaf=63, l2_regularization=0.239, max_leaf_nodes=48`) at 0.7082 (~0.49%),
the strongest signal of any P3 candidate. `learning_rate` again landed near the new
floor (0.0039 vs. floor 0.003) - a further expansion was considered but the user
opted to confirm with a real submission first, following this project's now-standard
practice (a weak-or-strong-but-unconfirmed proxy signal gets checked for real before
more compute is spent chasing it further).

**Real Zindi submission** (`outputs/submission_hparams_candidate1.csv` - identical
production pipeline, only the model's hyperparameters changed): **0.758156 RMSE** -
**worse** than the current best (0.755129, P1+P2's pipeline with default
hyperparameters), despite the proxy's clear 0.49%/4-5-seed win. A genuine
proxy-real inversion, this time on the hyperparameter axis rather than a feature.

**Working hypothesis for the inversion** (not yet independently verified):
`HistGradientBoostingRegressor`'s built-in `early_stopping` carves its internal
validation split via a plain RANDOM 10% holdout from whatever training data it's
given - it has no way to respect the horizon-matched/mask-aware structure this
project's OWN validation harness was built specifically to get right (see
`notebooks/05_leaderboard_gap_investigation.ipynb`, `07_mask_aware_validation.ipynb`).
At a very low learning rate, the number of boosting rounds is unusually sensitive to
exactly when training stops - if the internal random split's stopping signal
diverges from what actually generalises to Test.csv's real (long-horizon-heavy,
partly-masked) deployment distribution, the model could stop at a rounds count
that looks good on our external proxy's *own* random draw but doesn't transfer.
This mirrors the project's foundational P0 finding (a validation split that pools
information differently than real deployment gives a misleading picture) applied
one level deeper, inside a library's own internals rather than this project's code.

**Gate decision: NOT graduated.** `src/model.py`'s hyperparameters are unchanged.
Logged as a negative result per `structuring-ml-projects` rule 7 - this project's
practice of confirming ambiguous-but-appealing proxy signals with a real submission
(established during P2) caught a case where the proxy signal was actually more
misleading than P2's, worth remembering: proxy wins are not more trustworthy just
because the change under test is "only" a hyperparameter rather than a new feature.
